In [2]:
import cv2
from config import *
import os
import numpy as np
from keras.models import Sequential, Model
from keras.layers import *
from sklearn.model_selection import train_test_split
from keras.optimizers import Adam
from tensorflow.keras.saving import register_keras_serializable
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow as tf
import gc
import random

In [13]:
list1=os.listdir(DATA_PATH)
healthy=[i for i in list1 if i[-7:]=='healthy']
not_healthy=[i for i in list1 if i[-7:]!='healthy']
category_list=[i.split("_")[0] for i in healthy]

for i in ["Blueberry___healthy","Soybean___healthy","Raspberry___healthy"]:
    healthy.remove(i)
for i in ["Blueberry","Soybean","Raspberry"]:
    category_list.remove(i)

temp_img=[]
temp_label=[]

for folder in healthy:
    inside_folder= os.path.join(DATA_PATH,folder)
    for i in range(3):
        temp_img.append((cv2.imread(os.path.join(inside_folder,random.choice(os.listdir(inside_folder))))).astype('float32')/255.0)
        temp_label.append(folder)

for i in category_list:
    folder=random.choice([j for j in not_healthy if j.split("_")[0]==i])
    inside_folder= os.path.join(DATA_PATH,folder)
    for k in range(3):
        temp_img.append((cv2.imread(os.path.join(inside_folder,random.choice(os.listdir(inside_folder))))).astype('float32')/255.0)
        temp_label.append(i+"__not_healthy")


In [14]:
img_list = np.array(temp_img)
label_list = np.array(temp_label)
x_train, x_test, y_train, y_test = train_test_split(img_list,label_list,test_size= 0.2, random_state = 4)
img_list.shape

(54, 256, 256, 3)

In [15]:
import itertools
def make_paired_dataset(X, y):
    X_pairs, y_pairs = [], []
    tuples = [(x1, y1) for x1, y1 in zip(X, y)]

    for t in itertools.product(tuples, tuples):

        pair_A, pair_B = t
        img_A, label_A = t[0]
        img_B, label_B = t[1]
        new_label = int(label_A == label_B)
        X_pairs.append([img_A, img_B])
        y_pairs.append(new_label)

    x_pairs = np.array(X_pairs)
    y_pairs = np.array(y_pairs)
  
    return x_pairs, y_pairs


# random_indices = np.random.choice(x_train.shape[0], 43, replace=True)
# x_train_sample, y_train_sample = x_train[random_indices], y_train [random_indices]
X_pairs, y_pairs = make_paired_dataset(x_train, y_train)

# random_indices = np.random.choice(x_test.shape[0],43,replace=True)
# x_test_sample , y_test_sample = x_test[random_indices], y_test[random_indices]
x_test_pair, y_test_pair = make_paired_dataset(x_test,y_test)

gc.collect()
del x_train, x_test, y_train, y_test

In [16]:
@register_keras_serializable(package="Custom", name="contrastive_loss")
def contrastive_loss(y_true, y_pred, margin = 0.1):
    positive_loss = y_true * tf.square(y_pred)
    negative_loss = (1-y_true)*tf.square(tf.maximum(margin - y_pred , 0))
    return tf.reduce_mean(positive_loss+negative_loss)

def build_model():
    img_left = Input((256, 256, 3), name = "img_left")
    img_right = Input((256, 256, 3), name = "img_right")

    cnn = Sequential()
    cnn.add(Reshape((256, 256, 3)))
    cnn.add(Conv2D(64, (5, 5), padding = "same"))
    cnn.add(BatchNormalization())
    cnn.add(MaxPooling2D())
    cnn.add(ReLU())
    cnn.add(Conv2D(64, (5, 5), padding = "same"))
    cnn.add(BatchNormalization())
    cnn.add(MaxPooling2D())
    cnn.add(ReLU())
    cnn.add(GlobalAveragePooling2D())
    
    # cnn.add(Dense(64, activation = "relu"))

    feature_left  = cnn(img_left)
    feature_right = cnn(img_right)

    # distance = Lambda(lambda tensors:abs(tensors[0] - tensors [1]))([feature_left, feature_right])
    difference_layer = Subtract()([feature_left, feature_right])
    output = Dense(1, activation = "sigmoid")(difference_layer)

    model = Model(inputs=[img_left, img_right], outputs = output)
    model.compile(optimizer = Adam(learning_rate = 0.001), loss = contrastive_loss, metrics = ['accuracy'])
    return model

def train_model(model, x_pair, y_pair, x_test_pair, y_test_pair):
    early_stopping = EarlyStopping(monitor = 'val_accuracy', patience = 3, restore_best_weights = True)

    # Train the model
    model.fit(x = [x_pair[:, 0, :, :], x_pair[:, 1, :, :]],
                    y = y_pair,
                    validation_data = ([x_test_pair[:, 0, :, :], x_test_pair[:, 1, :, :]],y_test_pair),
                    epochs = 5, 
                    batch_size = 16,
                    callbacks=[early_stopping])
    
def evaluate_model(model, X_test_pad, y_test):
    loss, accuracy = model.evaluate(
    [X_test_pad[:, 0], X_test_pad[:, 1]], 
    y_test
    )

    print(f"Test Loss: {loss:.4f}, Test Accuracy: {accuracy:.4f}")

In [82]:
model = build_model()
model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ img_left            │ (None, 256, 256,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ img_right           │ (None, 256, 256,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential_1        │ (None, 64)        │    107,840 │ img_left[0][0],   │
│ (Sequential)        │                   │            │ img_right[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ subtract_1          │ (None, 64)        │          0 │ sequential_1[0][… │
│ (Subtract)          │                   │            │ sequential_1[1][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 1)         │         65 │ subtract_1[0][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 107,905 (421.50 KB)

 Trainable params: 107,649 (420.50 KB)

 Non-trainable params: 256 (1.00 KB)

In [83]:
tf.config.run_functions_eagerly(True)
train_model(model, X_pairs, y_pairs, x_test_pair, y_test_pair)

c:\Users\DELL\AppData\Local\Programs\Python\Python311\Lib\site-packages\tensorflow\python\data\ops\structured_function.py:258: UserWarning: Even though the `tf.config.experimental_run_functions_eagerly` option is set, this option does not apply to tf.data functions. To force eager execution of tf.data functions, please use `tf.data.experimental.enable_debug_mode()`.
  warnings.warn(


Epoch 1/5


c:\Users\DELL\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\models\functional.py:225: UserWarning: The structure of `inputs` doesn't match the expected structure: ['img_left', 'img_right']. Received: the structure of inputs=('*', '*')
  warnings.warn(


116/116 ━━━━━━━━━━━━━━━━━━━━ 290s 2s/step - accuracy: 0.5363 - loss: 0.0170 - val_accuracy: 0.6281 - val_loss: 0.0247
Epoch 2/5
116/116 ━━━━━━━━━━━━━━━━━━━━ 318s 3s/step - accuracy: 0.5847 - loss: 0.0153 - val_accuracy: 0.6116 - val_loss: 0.0230
Epoch 3/5
116/116 ━━━━━━━━━━━━━━━━━━━━ 315s 3s/step - accuracy: 0.7834 - loss: 0.0144 - val_accuracy: 0.9008 - val_loss: 0.0206
Epoch 4/5
116/116 ━━━━━━━━━━━━━━━━━━━━ 294s 3s/step - accuracy: 0.7450 - loss: 0.0107 - val_accuracy: 0.8678 - val_loss: 0.0189
Epoch 5/5
116/116 ━━━━━━━━━━━━━━━━━━━━ 263s 2s/step - accuracy: 0.8531 - loss: 0.0092 - val_accuracy: 0.8926 - val_loss: 0.0172


In [ ]:
model.save("disease_detector.keras")

In [ ]:
from tensorflow.keras.models import load_model
import cv2
import numpy as np
import tensorflow as tf

# Load the saved model
model = load_model("disease_detector.keras", custom_objects={"contrastive_loss": contrastive_loss})

In [23]:
def preprocess_image(image_path):
    img = cv2.imread(image_path)
    img = cv2.resize(img, (256, 256))
    img = img.astype('float32') / 255.0
    return img

In [24]:
def predict_similarity(image_path1, image_path2):
    img1 = preprocess_image(image_path1)
    img2 = preprocess_image(image_path2)

    img1 = np.expand_dims(img1, axis=0)
    img2 = np.expand_dims(img2, axis=0)

    # Get similarity score
    prediction = model.predict([img1, img2])[0][0]

    # Print result
    if prediction < 0.5:
        print("❌ The plant is likely **diseased**.")
    else:
        print("✅ The plant is likely **healthy**.")

    return prediction


In [26]:
img1 = r"New Plant Diseases Dataset(Augmented)\New Plant Diseases Dataset(Augmented)\train\Apple___Apple_scab\0a5e9323-dbad-432d-ac58-d291718345d9___FREC_Scab 3417_270deg.JPG"
img2 = r"New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/train/Apple___healthy/0a285c8b-1c31-48d4-89f2-af8b9edc36f6___RS_HL 5759.JPG"

score = predict_similarity(img1, img2)
print(f"Similarity Score: {score:.4f}")


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
❌ The plant is likely **diseased**.
Similarity Score: 0.4847
